# Lecture 1 · Generative and agentic AI (Fable edition)

Companion notebook, generated from the code on the lecture pages (https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/). Run the cells in order; each one corresponds to a numbered step on the pages. Steps marked 💳 need `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `LLM_MODEL`) in the environment and are skipped otherwise.

## Step 1 — Split the request into three jobs, and ask a small model to do the arithmetic

[1. Take a request apart: generate, decide, calculate](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/01-take-a-request-apart/)

In [ ]:
import datetime, json, pathlib, urllib.request
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

REQUEST = (
    "I missed the lecture on attention — what is it, and how many days "
    "do I have "
    "until the Lecture 2 quiz on 14 November?")
# fixed, so every run prints the same numbers
TODAY = datetime.date(2026, 10, 31)

# Job 1, decide: which lecture? A keyword rule stands in for the router Lecture
# 8 trains.
route = ("Lecture 1 · Generative "
         "and agentic AI") if "attention" in REQUEST.lower() else "unknown"
# Job 2, calculate: a date difference is arithmetic, so code does it.
days = (datetime.date(2026, 11, 14) - TODAY).days
print("decide   ->", route)
print("calculate->", days, "days")

# Job 3, generate: a small language model (135 M parameters, CPU, float32 —
# Lecture 8, 5.2).
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32)
torch.manual_seed(0)

def ask(question, max_new_tokens=40):
    """Ask the model one question; return its reply as plain text."""
    text = tok.apply_chat_template([{"role": "user", "content": question}],
                                   add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    out = model.generate(ids, attention_mask=torch.ones_like(ids),
                         max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

print("generate ->",
      ask("In one sentence: what is attention in a language model?"))
print(
    "model asked to calculate ->",
    ask(
        "Today is 31 October 2026. How many days are left until "
        "14 November 2026? "
        "Reply with the number only.", 30))

## Step 2 — Split one sentence of the notes into tokens, in English and in Ukrainian

[2. How a language model writes, one token at a time](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/02-how-a-language-model-writes/)

In [ ]:
import re

CORPUS = ("https://aiml-study-guides.onrender.com/corpus/generative-agentic-"
          "ai/lectures-01-06.{}.jsonl")
notes = {}
for lang in ("en", "uk"):
    path = pathlib.Path(f"lectures-01-06.{lang}.jsonl")
    if not path.exists():
        urllib.request.urlretrieve(CORPUS.format(lang), path)
    rows = [json.loads(line) for line in path.open(encoding="utf-8")]
    # e.g. (1, "2.1")
    notes[lang] = {(r["lecture"], r["section"]): r["text"]
                   for r in rows if r["section"]}

for lang in ("en", "uk"):  # the same sentence of the notes in both languages
    sentence = notes[lang][(1, "2.1")].split("?")[0] + "?"
    pieces = tok.tokenize(sentence)
    print(f"{lang}: {len(re.findall(r'\w+', sentence)):>2} words -> "
          f"{len(pieces):>2} tokens | {sentence}")
    print("    first tokens:",
          [tok.convert_tokens_to_string([p]) for p in pieces[:10]])
print("the request:", len(re.findall(r"\w+", REQUEST)), "words ->",
      len(tok.tokenize(REQUEST)), "tokens")

## Step 3 — Ask the model for the probabilities of the next token

[2. How a language model writes, one token at a time](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/02-how-a-language-model-writes/)

In [ ]:
@torch.no_grad()
def next_token_probs(text, k=5):
    ids = tok(text, return_tensors="pt")["input_ids"]
    # one score per vocabulary entry -> probabilities
    probs = model(ids).logits[0, -1].softmax(-1)
    top = probs.topk(k)
    return [(tok.decode([int(i)]), float(p))
            for p, i in zip(top.values, top.indices)]

PROMPTS = ["A language model writes its answer one",
           "Attention lets each token look at the tokens",
           "Diffusion models create an image by removing"]
print("vocabulary size:", len(tok))
for text in PROMPTS:
    print(f"\n{text} ...")
    for piece, p in next_token_probs(text):
        print(f"   {piece!r:>14}  {p:.3f}")

## Step 4 — Continue one prompt greedily and by sampling at three temperatures

[2. How a language model writes, one token at a time](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/02-how-a-language-model-writes/)

In [ ]:
@torch.no_grad()
def continue_text(text, temperature=None, n_tokens=12, seed=0):
    torch.manual_seed(seed)
    ids = tok(text, return_tensors="pt")["input_ids"]
    sampling = dict(do_sample=True, temperature=temperature,
                    top_k=0) if temperature else dict(do_sample=False)
    out = model.generate(ids, attention_mask=torch.ones_like(ids),
                         max_new_tokens=n_tokens, **sampling)
    return tok.decode(out[0, ids.shape[1]:],
                      skip_special_tokens=True).replace("\n", " ")

prompt = PROMPTS[1]
print("greedy (always the most likely token):")
print("   ", continue_text(prompt))
for t in (0.3, 1.0, 1.8):
    print(f"sampling at temperature {t}:")
    for seed in range(3):
        print("   ", continue_text(prompt, temperature=t, seed=seed))

## Step 5 — Score a sentence of the notes and the same words shuffled

[2. How a language model writes, one token at a time](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/02-how-a-language-model-writes/)

In [ ]:
import math
import random

@torch.no_grad()
def mean_loss(text):
    ids = tok(text, return_tensors="pt")["input_ids"]
    # mean of -log p(next token) over the text
    return model(ids, labels=ids).loss.item()

# the sentence after the question
sentence = notes["en"][(1, "2.1")].split("?")[1].strip().split("\n")[0]
words = sentence.split()
random.seed(0); shuffled = " ".join(random.sample(words, len(words)))
for name, text in (("notes sentence", sentence), ("same words shuffled",
                                                  shuffled)):
    loss = mean_loss(text)
    print(f"{name:<20} loss {loss:5.2f}   perplexity "
          f"{math.exp(loss):8.1f} | {text}")

## Step 6 — Read the attention weights of the token “it”

[2. How a language model writes, one token at a time](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/02-how-a-language-model-writes/)

In [ ]:
attn_model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32,
                                                  attn_implementation="eager")
text = "The student closed the laptop because it was overheating."
ids = tok(text, return_tensors="pt")["input_ids"]
with torch.no_grad():
    # one tensor per layer: [batch, head, query, key]
    att = attn_model(ids, output_attentions=True).attentions
pieces = [tok.decode([int(i)]) for i in ids[0]]
q = pieces.index(" it")
print(f"{len(att)} layers x {att[0].shape[1]} heads; row of the token ' it' "
      f"(position {q}), weights on earlier tokens")
for layer in (0, 15, 29):
    for head in (0, 4, 8):
        w = att[layer][0, head, q, :q + 1]
        top = w.topk(3)
        print(f"layer {layer:>2} head {head}: " +
              ", ".join(f"{pieces[int(i)]!r} {float(v):.2f}"
                        for v, i in zip(top.values, top.indices)))
mean = torch.stack([a[0, :, q, :q + 1].mean(0) for a in att]).mean(0)
print("averaged over all heads and layers:",
      ", ".join(f"{p!r} {float(v):.2f}" for p, v in zip(pieces, mean)))

## Step 7 — Compare the memory per token of SmolLM2 with Mamba’s fixed state

[2. How a language model writes, one token at a time](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/02-how-a-language-model-writes/)

In [ ]:
from transformers import AutoConfig

c = AutoConfig.from_pretrained(BASE)
head_dim = c.hidden_size // c.num_attention_heads
# keys + values, float32
kv_bytes = 2 * c.num_hidden_layers * c.num_key_value_heads * head_dim * 4
print(f"SmolLM2-135M: {c.num_hidden_layers} layers, {c.num_key_value_heads} "
      f"key/value heads of {head_dim} -> "
      f"{kv_bytes:,} bytes cached per token")
for n in (1_000, 8_192):  # 8,192 = its longest context
    print(
        f"   {n:>7,} tokens of context: {n * kv_bytes / 1e6:8.0f} MB of "
        f"keys and values")

m = AutoConfig.from_pretrained("state-spaces/mamba-130m-hf")
inner = m.expand * m.hidden_size
# SSM state + conv buffer, float32
state = m.num_hidden_layers * inner * (m.state_size + m.conv_kernel - 1) * 4
print(f"Mamba-130M: {m.num_hidden_layers} layers, inner width {inner}, state "
      f"size {m.state_size} -> "
      f"{state / 1e6:.1f} MB of state, the same at any context length")

## Step 8 — Noise toy data step by step, then generate by denoising

[3. How an image model paints: from noise to a picture](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/03-how-an-image-model-paints/)

In [ ]:
import numpy as np

# Toy "images": 2-D points on two shapes — a ring (class 0) and a square
# outline (class 1), densely sampled.
rng = np.random.default_rng(0)
a = rng.uniform(0, 2 * np.pi, 2000)
ring = np.c_[np.cos(a), np.sin(a)] * 2.0
s, side = rng.uniform(-1, 1, 2000), rng.integers(0, 4, 2000)
square = 0.8 * np.where(side[:, None] < 2,
                        np.c_[s, np.where(side == 0, 1.0, -1.0)],
                        np.c_[np.where(side == 2, 1.0, -1.0), s])
data, labels = np.r_[ring, square], np.r_[np.zeros(2000, int),
                                          np.ones(2000, int)]

T = 100
# share of the signal left after t noising steps
alpha_bar = np.cumprod(1 - np.linspace(1e-4, 0.1, T))

def denoise(x, t, pts=data):
    """Best guess of the clean point behind noisy x: a likelihood-weighted
    average of the shape's points."""
    d2 = ((x[:, None, :] - np.sqrt(alpha_bar[t]) * pts[None])**
          2).sum(-1) / (2 * (1 - alpha_bar[t]))
    w = np.exp(-(d2 - d2.min(1, keepdims=True)))
    return (w / w.sum(1, keepdims=True)) @ pts

def sample(n, steps, guess=denoise, seed=1):
    # start from pure noise
    x = np.random.default_rng(seed).normal(size=(n, 2))
    ts = np.linspace(T - 1, 0, steps).astype(int)
    for i, t in enumerate(ts):
        x0 = guess(x, t)
        # the noise it believes is in x
        eps = (x - np.sqrt(alpha_bar[t]) * x0) / np.sqrt(1 - alpha_bar[t])
        nxt = alpha_bar[ts[i + 1]] if i + 1 < len(ts) else 1.0
        # one step back toward clean data
        x = np.sqrt(nxt) * x0 + np.sqrt(1 - nxt) * eps
    return x

# mean distance from each sample to the nearest point of either shape
def off_shape(x):
    return np.sqrt(((x[:, None] - data[None]) ** 2).sum(-1)).min(1).mean()

# share of samples closer to the ring than to the square (the data is half and
# half)
def on_ring(x):
    return labels[np.sqrt(((x[:, None] - data[None])**
                           2).sum(-1)).argmin(1)].tolist().count(0) / len(x)

print(
    f"signal left after {T} noising steps: {np.sqrt(alpha_bar[-1]):.3f} (the "
    f"rest is noise)")
for steps in (1, 2, 5, 20, 100):
    x = sample(300, steps)
    print(f"reverse process in {steps:>3} steps: mean distance to the shapes "
          f"{off_shape(x):.3f}, "
          f"on the ring {on_ring(x):.0%}")

## Step 9 — Ask for the ring: classifier-free guidance and a negative prompt

[3. How an image model paints: from noise to a picture](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/03-how-an-image-model-paints/)

In [ ]:
# how strongly the model's own conditional guess follows the prompt (weakened
# on purpose)
FOLLOW = 0.3

def guided(want, scale, avoid=None):
    """Classifier-free guidance: start from the guess without a prompt (or
    the negative prompt's guess) and move `scale` times along the direction
    the prompt points to."""
    def guess(x, t):
        base = denoise(x, t) if avoid is None else denoise(
            x, t, data[labels == avoid])
        cond = FOLLOW * denoise(
            x, t, data[labels == want]) + (1 - FOLLOW) * denoise(x, t)
        return base + scale * (cond - base)
    return guess

for scale in (0.0, 1.0, 2.0, 3.0):
    print(f"prompt 'ring', guidance scale {scale}: on the ring "
          f"{on_ring(sample(300, 20, guided(0, scale))):.0%}")
for scale in (1.2, 1.5):
    print(f"scale {scale}: on the ring "
          f"{on_ring(sample(300, 20, guided(0, scale))):.0%} without a "
          f"negative prompt, "
          f"{on_ring(sample(300, 20, guided(0, scale, avoid=1))):.0%} with "
          f"negative prompt 'square'")

## Step 10 — Count the tokens and attention entries of a Diffusion Transformer

[3. How an image model paints: from noise to a picture](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/03-how-an-image-model-paints/)

In [ ]:
# A Diffusion Transformer cuts the latent image into patches; every patch is
# one token for attention.
for pixels in (512, 1024):
    latent = pixels // 8  # a typical autoencoder shrinks each side 8 times
    for patch in (2, 4):
        tokens = (latent // patch) ** 2
        print(
            f"{pixels}x{pixels} image -> latent {latent}x{latent}, patch "
            f"{patch}: {tokens:>5,} tokens, "
            f"attention matrix {tokens ** 2:>11,} entries per head and layer")

## Step 11 — An agent loop with a scripted model

[5. Agents: systems that act toward a goal](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-01-fable/05-agents-systems-that-act/)

In [ ]:
# A minimal agent loop: the model proposes the next action, the application
# executes it and reports back.
reminders = []

def tool_days_until(date):        # calculate: code, not the model
    return (datetime.date.fromisoformat(date) - TODAY).days

def tool_add_reminder(text, date):
    datetime.date.fromisoformat(date)  # raises ValueError on a bad date
    reminders.append((date, text)); return f"saved ({len(reminders)} reminder)"

TOOLS = {"days_until": tool_days_until, "add_reminder": tool_add_reminder}

# A scripted model: it replays the decisions a real model would make, so the
# loop runs offline and the same every time.
SCRIPT = [
    {"tool": "days_until", "args": {"date": "2026-11-14"}},
    # a mistake
    {
        "tool": "add_reminder",
        "args": {"text": "Lecture 2 quiz", "date": "14 November"}
    },
    # corrected
    {
        "tool": "add_reminder",
        "args": {"text": "Lecture 2 quiz", "date": "2026-11-14"}
    },
    {
        "answer": "You have 14 days until the Lecture 2 quiz; I added a "
        "reminder for 14 November."
    }
]

def scripted_model(history):
    return SCRIPT[sum(1 for h in history if h["role"] == "tool")]

history = [{
    "role": "user",
    "content": "How many days until the Lecture 2 quiz on "
    "14 November? Remind me."
}]
# a hard limit: the loop must stop even if the model does not
for turn in range(6):
    decision = scripted_model(history)
    if "answer" in decision:
        print("answer:", decision["answer"]); break
    try:
        result = TOOLS[decision["tool"]](**decision["args"])
    except ValueError as err:
        # the error goes back to the model as feedback
        result = f"error: {err}"
    print(f"turn {turn}: {decision['tool']}({decision['args']}) -> {result}")
    history.append({"role": "tool", "content": str(result)})
print("reminders:", reminders)